# 03 - Tweedie GLM Model

## Purpose

The purpose of this notebook is to develop and evaluate a Tweedie generalized linear model for modelling commercial property fire-peril loss cost. The Tweedie distribution with a log link is used to accommodate the highly zero-inflated and right-skewed nature of the insurance loss data. The model serves as a traditional actuarial baseline against which more flexible machine-learning approaches are compared.

## Methodology

A Tweedie generalized linear model with a log link is fitted to model the fire-peril loss cost. The Tweedie distribution is selected because of its ability to accommodate a point mass at zero and a continuous positive tail. Model performance is evaluated using RMSE, MAE and R^2 on a test set.

In [1]:
# ======================================================
# CIND820 - Keystone Project
# Tweedie GLM Model: Fire Peril Loss Cost (Liberty Mutual Insurance)
# Author: Debra Allen 
# ======================================================

# Call Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

## Data Loading

In [2]:
# =======================================================
# Data Preparation for Modeling: Fire Peril Loss Cost
# =======================================================

# Import necessary libraries for data preparation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import TweedieRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
# ======================================================
# Load the dataset and display basic information
# ======================================================

# Use a smaller sample for modeling to avoid memory issues
def find_repo_root(start_path=None, marker_files=("'git", "README.md")):
    """
    Walk up the directory to find the repository root.
    The root is identified by the presence of marker files (e.g., .git, README.md).
    """
    if start_path is None:
        start_path = Path.cwd()
    
    start_path = start_path.resolve()

    for parent in [start_path] + list(start_path.parents):
        for marker in marker_files:
            if (parent / marker).exists():
                return parent
    raise FileNotFoundError("Could not find repo root. Run notebook from inside the repo.")

REPO_ROOT = find_repo_root()
SAMPLE_PATH = REPO_ROOT / "data" / "liberty_train_subset.csv"
print("Repo Root:", REPO_ROOT)
print("Data Path:", SAMPLE_PATH)

# Load a smaller sample of the dataset for modeling
model_data = pd.read_csv(SAMPLE_PATH, low_memory=False)

# Display basic information about the dataset
model_data.info()

Repo Root: C:\Users\uni_f\Downloads\LaTex\TMU-CIND820
Data Path: C:\Users\uni_f\Downloads\LaTex\TMU-CIND820\data\liberty_train_subset.csv
<class 'pandas.DataFrame'>
RangeIndex: 113015 entries, 0 to 113014
Columns: 302 entries, id to weatherVar236
dtypes: float64(291), int64(1), str(10)
memory usage: 260.4 MB


## Data Preparation

In [4]:
# ======================================================
# Prepare the dataset for modeling
# ======================================================

# Convert categorical variables to string type (if any)
for col in model_data.select_dtypes(include=["object"]).columns:
    model_data[col] = model_data[col].astype(str)

# Define the target variable and predictor variables
target = "target"
X = model_data.drop(columns=[target])
y = model_data[target]

C:\Users\uni_f\AppData\Local\Temp\ipykernel_7092\2822705644.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in model_data.select_dtypes(include=["object"]).columns:


In [5]:
# ======================================================
# Identify Variable Types
# ======================================================

# Identify categorical and numerical variables
categorical_vars = X.select_dtypes(include=["object"]).columns.tolist()
numerical_vars = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Display variable types
print("Categorical Variables:", categorical_vars)
print("Numerical Variables:", numerical_vars)

# Display number of numerical and categorical variables
print("\nNumber of Categorical Variables:", len(categorical_vars))
print("Number of Numerical Variables:", len(numerical_vars))

Categorical Variables: ['var1', 'var2', 'var3', 'var4', 'var5', 'var6', 'var7', 'var8', 'var9', 'dummy']
Numerical Variables: ['id', 'var10', 'var11', 'var12', 'var13', 'var14', 'var15', 'var16', 'var17', 'crimeVar1', 'crimeVar2', 'crimeVar3', 'crimeVar4', 'crimeVar5', 'crimeVar6', 'crimeVar7', 'crimeVar8', 'crimeVar9', 'geodemVar1', 'geodemVar2', 'geodemVar3', 'geodemVar4', 'geodemVar5', 'geodemVar6', 'geodemVar7', 'geodemVar8', 'geodemVar9', 'geodemVar10', 'geodemVar11', 'geodemVar12', 'geodemVar13', 'geodemVar14', 'geodemVar15', 'geodemVar16', 'geodemVar17', 'geodemVar18', 'geodemVar19', 'geodemVar20', 'geodemVar21', 'geodemVar22', 'geodemVar23', 'geodemVar24', 'geodemVar25', 'geodemVar26', 'geodemVar27', 'geodemVar28', 'geodemVar29', 'geodemVar30', 'geodemVar31', 'geodemVar32', 'geodemVar33', 'geodemVar34', 'geodemVar35', 'geodemVar36', 'geodemVar37', 'weatherVar1', 'weatherVar2', 'weatherVar3', 'weatherVar4', 'weatherVar5', 'weatherVar6', 'weatherVar7', 'weatherVar8', 'weatherVar9

C:\Users\uni_f\AppData\Local\Temp\ipykernel_7092\2312351895.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_vars = X.select_dtypes(include=["object"]).columns.tolist()


## Preprocessing the Data

In [6]:
# ======================================================
# Preprocess the dataset
# ======================================================

# Numerical pipeline: impute missing values with median and scale features
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: impute missing values with most frequent and one-hot encode
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine numerical and categorical pipelines into a ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numerical_vars),
    ('cat', categorical_pipeline, categorical_vars)
],
remainder='drop',
n_jobs=-1
)

In [7]:
# =======================================================
# Train-Test Split
# =======================================================

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining observations:", X_train.shape[0])
print("Testing observations:", X_test.shape[0])


Training observations: 90412
Testing observations: 22603


## Tweedie GLM Model

In [8]:
# ======================================================
# Tweedie GLM Model
# ======================================================

tweedie_glm = TweedieRegressor(
    power=1.5,  # Tweedie distribution with power between 1 and 2 for insurance claims
    alpha=0.1,  # Regularization strength
    link='log',
    max_iter=1000
)

In [9]:
# ======================================================
# Full Modeling Pipeline
# ======================================================

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', tweedie_glm)
])

## Model Training

In [10]:
# ======================================================
# Model Training
# ======================================================

model_pipeline.fit(X_train, y_train)

print("Tweedie GLM fitted successfully.")

Tweedie GLM fitted successfully.


## Cross-Validation

In [13]:
# ======================================================
# Cross-Validation for Tweedie GLM
# ======================================================

# Define K-Fold cross-validation
def cross_validate_rmse(model_pipeline, X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    rmse_scores = []
    
    for train_index, val_index in kf.split(X):
        X_train_cv, X_val_cv = X.iloc[train_index], X.iloc[val_index]
        y_train_cv, y_val_cv = y.iloc[train_index], y.iloc[val_index]
        
        model_pipeline.fit(X_train_cv, y_train_cv)
        y_pred_cv = model_pipeline.predict(X_val_cv)
        
        rmse = np.sqrt(mean_squared_error(y_val_cv, y_pred_cv))
        rmse_scores.append(rmse)
    
    return np.mean(rmse_scores), np.std(rmse_scores)

# Perform cross-validation and print results
cv_mean_glm, cv_std_glm = cross_validate_rmse(
    model_pipeline,
    X_train,
    y_train,
    n_splits=5
)

print(f"Cross-Validation Results for Tweedie GLM:")
print(f"Mean RMSE: {cv_mean_glm:.4f}", cv_mean_glm)
print(f"Standard Deviation: {cv_std_glm:.4f}", cv_std_glm)

Cross-Validation Results for Tweedie GLM:
Mean RMSE: 0.2039 0.20385202727942095
Standard Deviation: 0.0334 0.03342007233954466


## Model Evaluation

In [19]:
# ======================================================
# Model Evaluation
# ======================================================

# Predict on the test set
y_pred = model_pipeline.predict(X_test)

# Evaluate model performance
mse = mean_squared_error(y_test, y_pred)  # RMSE
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Display evaluation results
print("\nTweedie GLM Performance (Test Dataset):")
print("---------------------------------------")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-squared (R2): {r2:.4f}")


Tweedie GLM Performance (Test Dataset):
---------------------------------------
Mean Squared Error (MSE): 0.0626
Mean Absolute Error (MAE): 0.0144
R-squared (R2): -0.0001


## Coefficients Interpretation

In [21]:
# ======================================================
# Coefficients Interpretation
# ======================================================

# Extract model coefficients after preprocessing
var_names_num = numerical_vars
var_names_cat = model_pipeline.named_steps['preprocessor'] \
    .named_transformers_["cat"] \
    .named_steps["onehot"] \
    .get_feature_names_out(categorical_vars)

feature_names = np.concatenate([var_names_num, var_names_cat])

# Extract coefficients from the model
coefficients = model_pipeline.named_steps['model'].coef_

# Create a DataFrame for coefficients
coef_table = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', key=abs, ascending=False)

coef_table.head(20)  # Display top 20 features by absolute coefficient value

,Feature,Coefficient
4,var13,-0.185218
336,var4_M1,0.103273
1,var10,-0.099791
268,weatherVar214,0.096316
371,var8_1,0.094750
7,var16,-0.084189
156,weatherVar102,0.081940
274,weatherVar220,-0.072615
157,weatherVar103,0.072076
54,geodemVar37,0.069565


## Tweedie GLM Summary

A Tweedie generalized linear model with a log link was fitted to model fire-peril loss cost. The Tweedie distribution is well-suited to insurance data because it can handle a large amount of zeros and a continuous positive tail. The model evaluation on a hold-out test shows limited explanatory power, reflecting the extreme sparsity and weak linear relationships in the data. Nevertheless, the GLM provides a stable interpretable baseline against which is more flexible machine-learning methods can be compared.